[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SeyhaLite/ml-foundations/blob/main/docs/notebooks/03_dataset_prep.ipynb)

# ០៣ · ត្រៀម Dataset សម្រាប់ហ្វឹក

In [ ]:
# ⚙️ Setup — រត់ cell នេះមុនគេ (Auto-setup for Google Colab)
import os, subprocess, sys

if "google.colab" in sys.modules:
    os.chdir("/content")
    if not os.path.isdir("ml-foundations"):
        subprocess.run(["git", "clone", "-q",
                        "https://github.com/SeyhaLite/ml-foundations.git"], check=True)
    os.chdir("ml-foundations")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"], check=True)

# ធ្វើឱ្យ qwen_ft import បានភ្លាមៗក្នុង session នេះ (មិនបាច់ restart)
for _cand in ("src", os.path.join("..", "..", "src")):
    _cand = os.path.abspath(_cand)
    if os.path.isdir(_cand) and _cand not in sys.path:
        sys.path.insert(0, _cand)

import qwen_ft
print("qwen-ft", qwen_ft.__version__, "ready ✅")

## ទម្រង់ messages JSONL

កំណត់ត្រានីមួយៗ = សន្ទនាមួយ (JSON object មួយជួរ)។ យើងនឹង៖
1. សរសេរ dataset តូចមួយដោយដៃ
2. ផ្ទៀងផ្ទាត់វា
3. បំបែក train/eval ដោយ deterministic

In [ ]:
from qwen_ft.data import write_jsonl, validate_jsonl_file, read_jsonl, split_dataset

def m(role, content): return {"role": role, "content": content}

records = [
    {"messages": [m("user", "រាជធានីកម្ពុជា?"), m("assistant", "ភ្នំពេញ។")]},
    {"messages": [m("user", "១+១?"), m("assistant", "២")]},
    {"messages": [m("system", "ឆ្លើយខ្លីៗ"), m("user", "Python គឺជាអ្វី?"),
                  m("assistant", "ភាសាសរសេរកម្មវិធីមួយ។")]},
    {"messages": [m("user", "លេខ ៣ គុណ ៧?"), m("assistant", "២១")]},
    {"messages": [m("user", "សួស្តី"), m("assistant", "ជំរាបសួរ! ត្រូវការជំនួយអ្វី?")]},
    {"messages": [m("user", "តើ GPU ខុស CPU យ៉ាងណា?"),
                  m("assistant", "GPU មាន core ច្រើន សម្រាប់ការគណនា parallel។")]},
]
write_jsonl(records, "/tmp/mini_dataset.jsonl")
report = validate_jsonl_file("/tmp/mini_dataset.jsonl")
print(report)

## ផ្ទៀងផ្ទាត់ dataset ពិតរបស់ repo

In [ ]:
import os
# នៅលើ Colab: repo ត្រូវបាន clone ក្នុងថតបច្ចុប្បន្ន
path = "data/sample_khmer_assistant.jsonl"
if os.path.exists(path):
    report = validate_jsonl_file(path)
    print(f"✅ Dataset ខ្មែរគំរូ: {report['records']} កំណត់ត្រា")
    print(f"   ជុំសន្ទនា: {report['min_turns']}–{report['max_turns']} · មធ្យមភាគ {report['avg_chars']} អក្សរ")
else:
    print("⚠️ រត់ notebook នេះពី repo root (ឬ clone ជាមុនសិន)")

## បំបែក Train / Eval

`split_dataset()` ប្រើ seed → លទ្ធផលដដែលជានិច្ច (reproducible)។

In [ ]:
records = read_jsonl(path if os.path.exists(path) else "/tmp/mini_dataset.jsonl")
train, eval_ = split_dataset(records, eval_ratio=0.1, seed=42)
print(f"train: {len(train)} · eval: {len(eval_)}")

# ផ្ទៀងផ្ទាត់ថា seed ដូចគ្នា → លទ្ធផលដូចគ្នា
train2, eval2 = split_dataset(records, eval_ratio=0.1, seed=42)
assert train == train2 and eval_ == eval2
print("✅ deterministic split")

## កំហុសដែល validator ចាប់បាន

សាកល្បងកំណត់ត្រាខូច — សារកំហុសត្រូវតែមានប្រយោជន៍ (ប្រាប់ជួរ + មូលហេតុ):

In [ ]:
from qwen_ft.data import validate_record, DataValidationError

bad_cases = [
    {"messages": [m("user", "សួស្តី")]},                    # គ្មាន assistant
    {"messages": [m("user", "a"), m("user", "b"), m("assistant", "c")]},  # user ពីរជាប់
    {"messages": [m("bot", "x")]},                            # role ខុស
    {"prompt": "a", "completion": "b"},                       # ទម្រង់ខុស
]
for i, case in enumerate(bad_cases):
    try:
        validate_record(case)
        print(f"case {i}: ❌ មិនបានចាប់កំហុស!")
    except DataValidationError as e:
        print(f"case {i}: ✅ {e}")

## សង្ខេប + លំហាត់

1. JSONL messages + UTF-8 + `ensure_ascii=False`
2. ផ្ទៀងផ្ទាត់ជានិច្ចមុនហ្វឹក
3. Split deterministic ដោយ seed

**លំហាត់៖** បង្កើត dataset ១០ ឧទាហរណ៍អំពីប្រធានបទរបស់អ្នក ហើយផ្ទៀងផ្ទាត់។
បន្ទាប់មកអានជំពូក [៤ · ត្រៀមទិន្នន័យ](../chapters/ch04-datasets.md)។